In [0]:
-- ========================================
-- COMPARISON: Jordan Quarterly Metrics vs World Bank Data
-- ========================================
-- 
-- DATA SOURCES:
-- 1. Jordan Quarterly Metrics: Jordan Department of Statistics (DOS)
--    Source: https://jorinfo.dos.gov.jo/View_ind/ind_en.aspx
--    Coverage: ALL RESIDENTS (citizens + non-citizens including Syrian refugees)
--    Methodology: National Labor Force Survey (de facto population)
--    Frequency: Quarterly
--    
-- 2. World Bank: ILO Modeled Estimates
--    Source: World Bank World Development Indicators
--    Coverage: Citizens + long-term residents (modeled, may exclude/weight refugees differently)
--    Methodology: Econometric model harmonizing national data to ILO standards
--    Frequency: Annual
--
-- KEY FINDING: Post-2016 divergence (+3-5% unemployment gap) reflects
-- Syrian refugee crisis impact. Jordan DOS captures full labor market pressure;
-- World Bank ILO estimates use standardized modeling for international comparability.
-- ========================================
-- 4: unemployment, youth unemployment, inflation, gdp growth, 
SELECT * 
FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
WHERE year >= 2020
  AND metric = 'unemployment'
ORDER BY 1 DESC, 2 DESC 

-- unemployment total, unemployment youth total, inflation, gdp growth 
SELECT year, 
  indicator_name,
  value
FROM info_env_jordan.bronze.worldbank_jordan_long
WHERE year >= 2020
ORDER BY 1 DESC, 2 DESC 


-- Check quarterly metrics coverage
SELECT 
  'Quarterly Metrics' AS source,
  MIN(year) AS earliest_year,
  MAX(year) AS latest_year,
  COUNT(DISTINCT year) AS year_count,
  COUNT(DISTINCT metric) AS metric_count
FROM info_env_jordan.bronze.jordan_econ_metrics_qtr;

-- Check World Bank coverage
SELECT 
  'World Bank' AS source,
  MIN(year) AS earliest_year,
  MAX(year) AS latest_year,
  COUNT(DISTINCT year) AS year_count,
  COUNT(DISTINCT indicator_name) AS indicator_count
FROM info_env_jordan.bronze.worldbank_jordan_long;
 
 
 -- Find overlapping unemployment & inflation indicators in World Bank
SELECT 
  indicator_name,
  indicator_code,
  COUNT(DISTINCT year) AS years_available,
  MIN(year) AS first_year,
  MAX(year) AS last_year
FROM info_env_jordan.bronze.worldbank_jordan_long
WHERE LOWER(indicator_name) LIKE '%unemployment%'
   OR LOWER(indicator_name) LIKE '%inflation%'
   OR LOWER(indicator_name) LIKE '%gdp%'
GROUP BY indicator_name, indicator_code
ORDER BY indicator_name;



-- Compare unemployment rates (aggregate quarterly to annual)
WITH quarterly_annual AS (
  SELECT 
    year,
    AVG(CAST(value AS DOUBLE)) AS avg_unemployment_qtr
  FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
  WHERE metric = 'unemployment'
  GROUP BY year
),
worldbank_unemployment AS (
  SELECT 
    year,
    value AS unemployment_wb,
    indicator_name
  FROM info_env_jordan.bronze.worldbank_jordan_long
  WHERE indicator_code = 'SL.UEM.TOTL.ZS'
)
SELECT 
  COALESCE(q.year, w.year) AS year,
  q.avg_unemployment_qtr AS quarterly_avg,
  w.unemployment_wb AS worldbank_annual,
  ROUND(q.avg_unemployment_qtr - w.unemployment_wb, 2) AS difference,
  w.indicator_name
FROM quarterly_annual q
FULL OUTER JOIN worldbank_unemployment w ON q.year = w.year
WHERE COALESCE(q.year, w.year) >= '2010'
ORDER BY year DESC;

-- Compare inflation rates
WITH quarterly_annual AS (
  SELECT 
    year,
    AVG(CAST(value AS DOUBLE)) AS avg_inflation_qtr
  FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
  WHERE metric = 'inflation_avg_qtr'
  GROUP BY year
),
worldbank_inflation AS (
  SELECT 
    year,
    value AS inflation_wb,
    indicator_name
  FROM info_env_jordan.bronze.worldbank_jordan_long
  WHERE indicator_code = 'FP.CPI.TOTL.ZG'
)
SELECT 
  COALESCE(q.year, w.year) AS year,
  q.avg_inflation_qtr AS quarterly_avg,
  w.inflation_wb AS worldbank_annual,
  ROUND(q.avg_inflation_qtr - w.inflation_wb, 2) AS difference,
  w.indicator_name
FROM quarterly_annual q
FULL OUTER JOIN worldbank_inflation w ON q.year = w.year
WHERE COALESCE(q.year, w.year) >= '2010'
ORDER BY year DESC;

-- Data recency comparison
SELECT 
  'Quarterly Metrics' AS source,
  'unemployment' AS metric,
  MAX(year) AS latest_year,
  COUNT(DISTINCT year) AS years_with_data
FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
WHERE metric = 'unemployment'
UNION ALL
SELECT 
  'World Bank',
  'unemployment',
  MAX(year),
  COUNT(DISTINCT year)
FROM info_env_jordan.bronze.worldbank_jordan_long
WHERE indicator_code = 'SL.UEM.TOTL.ZS'
UNION ALL
SELECT 
  'Quarterly Metrics',
  'inflation',
  MAX(year),
  COUNT(DISTINCT year)
FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
WHERE metric = 'inflation_avg_qtr'
UNION ALL
SELECT 
  'World Bank',
  'inflation',
  MAX(year),
  COUNT(DISTINCT year)
FROM info_env_jordan.bronze.worldbank_jordan_long
WHERE indicator_code = 'FP.CPI.TOTL.ZG';


-- ========================================
-- ANALYSIS: 2016 DIVERGENCE INVESTIGATION
-- ========================================

-- What happened in 2016? The unemployment gap pattern:
-- Pre-2016: Nearly perfect alignment (avg gap -0.03%)
-- 2016: Perfect match (0% gap) - LAST YEAR OF ALIGNMENT
-- Post-2016: Massive divergence (avg gap +3.33%, up to +5.36%)

-- Year-by-year breakdown showing the divergence
WITH quarterly_annual AS (
  SELECT 
    year,
    AVG(CAST(value AS DOUBLE)) AS unemployment_qtr,
    AVG(CASE WHEN metric = 'youth_unemployment' THEN CAST(value AS DOUBLE) END) AS youth_unemployment_qtr
  FROM info_env_jordan.bronze.jordan_econ_metrics_qtr
  WHERE metric IN ('unemployment', 'youth_unemployment')
  GROUP BY year
),
worldbank_data AS (
  SELECT 
    year,
    MAX(CASE WHEN indicator_code = 'SL.UEM.TOTL.ZS' THEN value END) AS unemployment_wb,
    MAX(CASE WHEN indicator_code = 'SL.UEM.1524.ZS' THEN value END) AS youth_unemployment_wb
  FROM info_env_jordan.bronze.worldbank_jordan_long
  WHERE indicator_code IN ('SL.UEM.TOTL.ZS', 'SL.UEM.1524.ZS')
  GROUP BY year
)
SELECT 
  COALESCE(q.year, w.year) AS year,
  q.unemployment_qtr AS quarterly_unemployment,
  w.unemployment_wb AS worldbank_unemployment,
  ROUND(q.unemployment_qtr - w.unemployment_wb, 2) AS unemployment_gap,
  q.youth_unemployment_qtr AS quarterly_youth_unemp,
  w.youth_unemployment_wb AS worldbank_youth_unemp,
  ROUND(q.youth_unemployment_qtr - w.youth_unemployment_wb, 2) AS youth_unemp_gap,
  CASE 
    WHEN COALESCE(q.year, w.year) < '2016' THEN 'Pre-2016 (Aligned)'
    WHEN COALESCE(q.year, w.year) = '2016' THEN '2016 (Divergence Starts)'
    ELSE 'Post-2016 (Diverged)'
  END AS period
FROM quarterly_annual q
FULL OUTER JOIN worldbank_data w ON q.year = w.year
WHERE COALESCE(q.year, w.year) BETWEEN '2012' AND '2025'
ORDER BY year;